# IDM-VTON Kaggle GPU API

Run on Kaggle with **GPU enabled** and **Internet on**. The final cell prints a public `/tryon` URL for your ASP.NET MVC app.

In [ ]:
import torch

!nvidia-smi
print('CUDA available:', torch.cuda.is_available())
print('Torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU from Kaggle Notebook Settings before continuing.')

In [ ]:
from pathlib import Path
import os
import subprocess

WORK = Path('/kaggle/working')
SPACE_DIR = WORK / 'IDM-VTON'

if not SPACE_DIR.exists():
    subprocess.check_call(['git', 'clone', 'https://huggingface.co/spaces/yisol/IDM-VTON', str(SPACE_DIR)])

os.chdir(SPACE_DIR)
print('Working directory:', os.getcwd())

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])

packages = [
    'diffusers==0.25.0',
    'transformers==4.36.2',
    'accelerate==0.26.1',
    'huggingface_hub==0.25.0',
    'einops==0.7.0',
    'onnxruntime>=1.22.0',
    'fvcore',
    'iopath',
    'cloudpickle',
    'omegaconf',
    'pycocotools',
    'av',
    'config==0.5.1',
    'fastapi',
    'uvicorn',
    'python-multipart',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
subprocess.check_call(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/kaggle/working/cloudflared'])
Path('/kaggle/working/cloudflared').chmod(0o755)
print('cloudflared ready:', os.path.getsize('/kaggle/working/cloudflared'), 'bytes')

In [ ]:
import importlib
import numpy
import torch
import torchvision
import cv2
import PIL

print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('cuda', torch.cuda.is_available())
print('numpy', numpy.__version__)
print('cv2', cv2.__version__)
print('PIL', PIL.__version__)
for name in ['diffusers', 'transformers', 'huggingface_hub', 'av', 'fvcore', 'omegaconf']:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'ok'))

In [ ]:
from pathlib import Path

app_path = Path('/kaggle/working/IDM-VTON/app.py')
text = app_path.read_text()
ui_start = text.find('garm_list = os.listdir')
if ui_start != -1:
    text = text[:ui_start] + '\nprint("IDM-VTON loaded in API mode")\n'
app_path.write_text(text)

spaces_path = Path('/kaggle/working/IDM-VTON/spaces.py')
spaces_path.write_text('''\ndef GPU(func=None, **kwargs):\n    def decorator(f):\n        return f\n    return decorator(func) if func is not None else decorator\n''')

gradio_path = Path('/kaggle/working/IDM-VTON/gradio.py')
gradio_path.write_text('''\nclass _Dummy:\n    def __init__(self, *args, **kwargs): pass\n    def __enter__(self): return self\n    def __exit__(self, *args): return False\n    def __call__(self, *args, **kwargs): return self\n    def queue(self, *args, **kwargs): return self\n\nBlocks = Row = Column = Accordion = _Dummy\nMarkdown = ImageEditor = Checkbox = Examples = Image = Textbox = Button = Slider = _Dummy\n''')

print('Patched app.py and added dummy spaces/gradio modules for Kaggle.')

In [ ]:
import importlib
import os
import sys
import torch

os.chdir('/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON/densepose')
sys.path.insert(0, '/kaggle/working/IDM-VTON/preprocess/openpose')

import diffusers.models.embeddings as embeddings
if not hasattr(embeddings, 'PositionNet'):
    embeddings.PositionNet = embeddings.GLIGENTextBoundingboxProjection

torch.cuda.empty_cache()
idm_app = importlib.import_module('app')
print('IDM-VTON model loaded.')

In [ ]:
%%writefile /kaggle/working/idm_vton_api.py
import base64
import io
import importlib
import os
import sys

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

os.chdir('/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON')
sys.path.insert(0, '/kaggle/working/IDM-VTON/densepose')
sys.path.insert(0, '/kaggle/working/IDM-VTON/preprocess/openpose')
idm_app = importlib.import_module('app')

api = FastAPI(title='IDM-VTON Kaggle API')
api.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

def normalize_category(value: str) -> str:
    value = (value or 'upper').strip().lower()
    if value in {'pants', 'lower', 'lower_body'}:
        return 'pants'
    if value in {'dress', 'dresses', 'overall'}:
        return 'dress'
    return 't-shirt'

def garment_description(category: str) -> str:
    normalized = normalize_category(category)
    if normalized == 'pants':
        return 'pants'
    if normalized == 'dress':
        return 'dress'
    return 'black short sleeve round neck t-shirt with front graphic print'

@api.get('/health')
def health():
    return {'status': 'ok', 'engine': 'idm-vton-kaggle'}

@api.post('/tryon')
async def tryon(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category: str = Form('upper'),
    denoise_steps: int = Form(30),
    seed: int = Form(42),
):
    person = Image.open(io.BytesIO(await person_image.read())).convert('RGB')
    garment = Image.open(io.BytesIO(await garment_image.read())).convert('RGB')
    result, _mask = idm_app.start_tryon(
        {'background': person, 'layers': [], 'composite': None},
        garment,
        garment_description(category),
        True,
        False,
        int(denoise_steps),
        int(seed),
    )
    output = io.BytesIO()
    result.save(output, format='PNG')
    return {'outputImageBase64': base64.b64encode(output.getvalue()).decode('utf-8')}


In [ ]:
import re
import subprocess
import threading
import time

server = subprocess.Popen([
    'python', '-m', 'uvicorn', 'idm_vton_api:api', '--host', '0.0.0.0', '--port', '8000'
], cwd='/kaggle/working')
time.sleep(8)

public_url = None
tunnel = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(160):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError('Cloudflare did not create a public URL. Run this cell again.')

print('\nHealth URL:', public_url + '/health')
print('Paste this into VirtualTryOn__ApiUrl:')
print(public_url + '/tryon')

def keep_logs_open():
    for line in tunnel.stdout:
        print(line, end='')

threading.Thread(target=keep_logs_open, daemon=True).start()